In [2]:
from BCP303.BCP303 import BPC303
from Sourcemeter.sourcemeter import Sourcemeter2401
import time
from datetime import datetime
from tool.tools import post_process


In [ ]:
# move the stage in Z direction
bcp303_z = BPC303(channel_id=1)
position_z = bcp303_z.move_to_origin(start_position=0)
step_size_z = 0.1  # move 0.1 mm
position_z = bcp303_z.bcp303_move_stage(step_size=step_size_z, current_position=position_z,)
bcp303_z.bcp303_stop(ifback=True)

In [12]:
# move the stage in x direction
bcp303_x = BPC303(channel_id=2)
position_x = bcp303_x.move_to_origin(start_position=0)
step_size_x = 0.1  # move 0.1 mm
position_x = bcp303_x.bcp303_move_stage(step_size=step_size_x, current_position=position_x,)
bcp303_x.bcp303_stop(ifback=False)

APT High Voltage Amplifier initialized
Stage Moving Done
Stage disconnected


In [ ]:
# record the signals
sm2401 = Sourcemeter2401(speed_nplc=0.1)


In [ ]:
# sweep operation: move stage in z-direction and record the signals

def sweep_operation(stage_settings=None,chip_name="chip_test",sample_name="beam_test"):
    step_number = stage_settings["step_number"]
    time_interval = stage_settings["time_interval"]
    position_x = stage_settings["position_x"]
    step_size_z = stage_settings["step_size_z"]
    count = 1
    formatted_time = datetime.now().strftime("%Y%m%d%H%M")
    try:
        bcp303_z = BPC303(channel_id=1)
        bcp303_device = bcp303_z.get_device()
        # moving controller x
        bcp303_x = BPC303(channel_id=2, device=bcp303_device)
        # Sourcemeter
        sm2401 = Sourcemeter2401(speed_nplc=0.1)
        # set the initial position of the stage z
        position_z = bcp303_z.move_to_origin(start_position=stage_settings["position_z"]) - step_size_z
        time.sleep(1)
        bcp303_x.move_to_origin()
        time.sleep(1)
        allData = [{"position": [], "voltage": []}]
        for _ in range(step_number):
            position_z = bcp303_z.bcp303_move_stage(
                step_size=step_size_z,
                current_position=position_z,
            )
            allData[0]["position"].append(position_z)
            time.sleep(0.5)
            bcp303_x.move_to_position(start_position=0, step_size=0.5, final_position=position_x)
            time.sleep(0.5)
            step_start = time.perf_counter()
            voltage = sm2401.measure_voltage(duration=time_interval / 2, dt=0.01)[
                "voltage"
            ]
            allData[0]["voltage"].append(voltage)
            # remaining time to wait until the next step
            elapsed = time.perf_counter() - step_start
            remaining = time_interval - elapsed
            if remaining > 0:
                time.sleep(remaining)
            bcp303_x.move_to_origin()
            time.sleep(0.5)
            print(f"process completed: {100 * count / step_number:.1f}%")
            count += 1
        sm2401_settings = sm2401.getSettings()
        stage_settings["position_x"] = position_x
        settings = {"stage": stage_settings, "sourcemeter": sm2401_settings}
        post_process(
            chip_name=chip_name,
            sample_name=sample_name,
            result=allData[0],
            config=settings,
            repeat=1,
            position_z=position_z,
            ifshow=False,
            formatted_time=formatted_time,
        )
    # stop the stage and sourcemeter
    except Exception as e:
        print(f"Exception occurred: {e}")
    finally:
        time.sleep(0.5)
        sm2401.close()
        bcp303_z.move_to_origin()
        bcp303_z.channel.StopPolling()
        bcp303_x.bcp303_stop(ifback=True)
        return allData[0], settings
    

setting = {
        "position_x": 1.4,
        "step_number": 40,
        "position_z": 0,
        "step_size_z": 0.5,
        "time_interval": 2,  
    }
sweep_operation(
        stage_settings=setting,
        chip_name="V1_R_W_1_Left",
        # chip_name="SiN_beam",
        # sample_name="w2_sweep",  # test_1_right, w=20
        sample_name="test_AFM4_450_w20_4_sweep",  
    )

APT High Voltage Amplifier initialized
APT High Voltage Amplifier initialized
IDN: KEITHLEY INSTRUMENTS INC.,MODEL 2401,4614233,B02 Jan 20 2021 10:19:49/B01  /W/N
process completed: 1.0%
process completed: 2.0%
process completed: 3.0%
process completed: 4.0%
process completed: 5.0%
process completed: 6.0%
process completed: 7.0%
process completed: 8.0%
process completed: 9.0%
process completed: 10.0%
process completed: 11.0%
process completed: 12.0%
process completed: 13.0%
process completed: 14.0%
process completed: 15.0%
process completed: 16.0%
process completed: 17.0%
process completed: 18.0%
process completed: 19.0%
process completed: 20.0%
process completed: 21.0%
process completed: 22.0%
process completed: 23.0%
process completed: 24.0%
process completed: 25.0%
process completed: 26.0%
process completed: 27.0%
process completed: 28.0%
process completed: 29.0%
process completed: 30.0%
process completed: 31.0%
process completed: 32.0%
process completed: 33.0%
process completed: 34

({'position': [0.0,
   0.189825128940703,
   0.388195440534684,
   0.588396862697226,
   0.788598284859768,
   0.988189336832789,
   1.18655964842677,
   1.38737144077883,
   1.58635212256233,
   1.78472243415632,
   1.98492385631886,
   2.1851252784814,
   2.38532670064394,
   2.58552812280648,
   2.7851191747795,
   2.98409985656301,
   3.18369090853603,
   3.38267159031953,
   3.58043153172399,
   3.77330851161229,
   3.97412030396435,
   4.1743217261269,
   4.37513351847896,
   4.5722830896939,
   4.76943266090884,
   4.9702444532609,
   5.16983550523392,
   5.36881618701743,
   5.56169316690573,
   5.76189458906827,
   5.96148564104129,
   6.16229743339335,
   6.36310922574542,
   6.56331064790796,
   6.7635120700705,
   6.96310312204352,
   7.16269417401654,
   7.36167485580004,
   7.56309701834162,
   7.76390881069368,
   7.96411023285623,
   8.16431165501877,
   8.36390270699179,
   8.56532486953337,
   8.76613666188543,
   8.96633808404798,
   9.1598254341258,
   9.35209204382